In [64]:
import pandas as pd
from pathlib import Path
import nfl_data_py as nfl

nfl_players = nfl.import_players()

nfl_teams = nfl.import_team_desc()


# Define path to file
file_path = BASE_DIR / "defender_attention_all.csv"

# Load CSV
df = pd.read_csv(file_path)

# Quick check
df.head()


# BASE_DIR is set in the config cell above

df_detailed_results = pd.read_csv(BASE_DIR / "intervention_detailed_results.csv")
df_overall_summary = pd.read_csv(BASE_DIR / "intervention_overall_summary.csv")
df_by_intervention_type = pd.read_csv(BASE_DIR / "intervention_by_type.csv")

df_temporal = pd.read_csv(BASE_DIR / "intervention_temporal_analysis.csv")
df_speed = pd.read_csv(BASE_DIR / "intervention_speed_sensitivity.csv")
df_direction = pd.read_csv(BASE_DIR / "intervention_direction_sensitivity.csv")
df_interactions = pd.read_csv(BASE_DIR / "intervention_pairwise_interactions.csv")

print("All CSVs loaded as individual DataFrames.")

supp_path = DATA_DIR / "supplementary_data.csv"
supp = pd.read_csv(supp_path)

All CSVs loaded as individual DataFrames.


In [ ]:
from pathlib import Path

# Set BASE_DIR to this repo root (or the folder where your data files live)
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"


In [69]:
import pandas as pd

# -------------------------------
# 1. Merge intervention data with play outcomes
# -------------------------------
df_eval = df_detailed_results.merge(
    supp,
    on=["game_id", "play_id"],
    how="inner"
)

# -------------------------------
# 2. Outcome metrics
# -------------------------------
df_eval["completion"] = (df_eval["pass_result"] == "C").astype(int)
df_eval["interception"] = (df_eval["pass_result"] == "INT").astype(int)
df_eval["touchdown"] = (df_eval["pass_result"] == "TD").astype(int)

# Directional passer rating proxy (consistent across plays)
df_eval["passer_rating_proxy"] = (
    1.5 * df_eval["completion"]
    + 0.05 * df_eval["yards_gained"].fillna(0)
    - 2.5 * df_eval["interception"]
    + 2.0 * df_eval["touchdown"]
)

# -------------------------------
# 3. Aggregate value by intervention type
# -------------------------------
agg_df = (
    df_eval
    .groupby("intervention_type")
    .agg(
        plays=("play_id", "count"),
        avg_impact_score=("impact_score", "mean"),
        avg_epa_allowed=("expected_points_added", "mean"),
        avg_yards_allowed=("yards_gained", "mean"),
        avg_passer_rating=("passer_rating_proxy", "mean")
    )
    .reset_index()
)

# -------------------------------
# 4. Correlation: impact vs outcomes (per intervention type)
# -------------------------------
corr_rows = []

for itype, sub in df_eval.groupby("intervention_type"):
    corr_rows.append({
        "intervention_type": itype,
        "corr_impact_epa": sub["impact_score"].corr(sub["expected_points_added"]),
        "corr_impact_yards": sub["impact_score"].corr(sub["yards_gained"]),
        "corr_impact_passer_rating": sub["impact_score"].corr(sub["passer_rating_proxy"])
    })

corr_df = pd.DataFrame(corr_rows)

# -------------------------------
# 5. Final evaluation table
# -------------------------------
intervention_evaluation = agg_df.merge(
    corr_df,
    on="intervention_type",
    how="left"
)

# -------------------------------
# 6. Defense-favorable ranking
# -------------------------------
intervention_evaluation["value_score"] = (
    -intervention_evaluation["avg_epa_allowed"]
    - 0.01 * intervention_evaluation["avg_yards_allowed"]
    - intervention_evaluation["avg_passer_rating"]
)

intervention_evaluation = intervention_evaluation.sort_values(
    "value_score",
    ascending=False
)

intervention_evaluation


,intervention_type,plays,avg_impact_score,avg_epa_allowed,avg_yards_allowed,avg_passer_rating,corr_impact_epa,corr_impact_yards,corr_impact_passer_rating,value_score
0,freeze,6417,-0.061734,0.334254,9.368396,1.42798,-0.014268,0.015555,-0.042665,-1.855919
1,misdirection,6417,-0.002087,0.334254,9.368396,1.42798,0.015262,0.011521,0.023307,-1.855919
2,removal,6417,0.005239,0.334254,9.368396,1.42798,-0.065182,-0.043253,-0.070206,-1.855919
3,slowdown,6417,-0.046585,0.334254,9.368396,1.42798,-0.007112,-0.008434,-0.006134,-1.855919


In [71]:
df_eval["high_impact"] = df_eval["impact_score"] >= df_eval["impact_score"].quantile(0.25)

df_eval.groupby(["intervention_type", "high_impact"]).agg(
    avg_epa=("expected_points_added", "mean"),
    avg_yards=("yards_gained", "mean"),
    plays=("play_id", "count")
)


avg_epa  avg_yards  plays
intervention_type high_impact                            
freeze            False        0.320998   8.973134   3350
                  True         0.348735   9.800130   3067
misdirection      False        0.337315  10.218182    165
                  True         0.334174   9.345969   6252
removal           False        0.374502  10.044492    472
                  True         0.331059   9.314718   5945
slowdown          False        0.343279   9.475720   2430
                  True         0.328754   9.302985   3987

In [105]:
import pandas as pd
import numpy as np
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from ipywidgets import Layout, HBox, VBox, Label
import warnings
warnings.filterwarnings('ignore')

def create_player_attention_table(df, nfl_players, nfl_teams, df_detailed_results=None):
    """
    Create an interactive player attention table with ipywidgets
    
    Parameters:
    -----------
    df : DataFrame
        Contains player attention metrics with columns:
        ['nfl_id', 'total_attention', 'avg_attention', 'max_attention', 
         'std_attention', 'median_attention', 'play_count', 'frame_count', 'high_attention_pct']
    
    nfl_players : DataFrame
        Contains player info with 'nfl_id' and other player details
    
    nfl_teams : DataFrame
        Contains team info with 'team_abbr' and color/logo information
    
    df_detailed_results : DataFrame, optional
        Contains intervention details with columns:
        ['game_id', 'play_id', 'defender_nfl_id', 'intervention_type', 
         'original_prediction', 'counterfactual_prediction', 'impact_score']
    """
    
    # ===== CRITICAL: Ensure nfl_id is consistent dtype across all dataframes =====
    df = df.copy()
    nfl_players = nfl_players.copy()
    nfl_teams = nfl_teams.copy()
    
    df['nfl_id'] = df['nfl_id'].astype(str)
    nfl_players['nfl_id'] = nfl_players['nfl_id'].astype(str)
    nfl_teams['team_abbr'] = nfl_teams['team_abbr'].astype(str)
    
    # Position grouping mapping
    position_groups = {
        'LB': ['LB', 'ILB', 'MLB', 'OLB'],
        'CB': ['CB', 'DB'],
        'S': ['S', 'SAF']
    }
    
    # Merge with player information
    df_with_players = df.merge(
        nfl_players[['nfl_id', 'display_name', 'position', 'latest_team', 'headshot']],
        on='nfl_id',
        how='left'
    )
    
    # Merge with team information
    df_with_players = df_with_players.merge(
        nfl_teams[['team_abbr', 'team_color', 'team_color2', 'team_logo_squared', 'team_wordmark']],
        left_on='latest_team',
        right_on='team_abbr',
        how='left'
    )
    
    # Add position group column
    def get_position_group(pos):
        if pd.isna(pos):
            return 'Other'
        for group, positions in position_groups.items():
            if any(p in str(pos) for p in positions):
                return group
        return 'Other'
    
    df_with_players['position_group'] = df_with_players['position'].apply(get_position_group)
    
    
    # Add intervention type counts if df_detailed_results is provided
    if df_detailed_results is not None and len(df_detailed_results) > 0:
        df_detailed_results = df_detailed_results[df_detailed_results['intervention_type'] == 'removal']
        df_detailed_results['defender_nfl_id'] = df_detailed_results['defender_nfl_id'].astype(str)
        
        # Count interventions by type for each player and calculate average impact score
        removal_stats = df_detailed_results.groupby('defender_nfl_id').agg(
            intervention_removal=('defender_nfl_id', 'size'),
            impact_removal=('impact_score', 'mean')).reset_index()
        

        df_with_players = df_with_players.merge(
            removal_stats, 
            left_on = 'nfl_id', 
            right_on = 'defender_nfl_id', 
            how = 'left')
        
        df_with_players['intervention_removal'] = df_with_players['intervention_removal'].fillna(0).astype(int)
        df_with_players['impact_removal'] = df_with_players['impact_removal'].fillna(0)
                
        
        # Get total intervention count
        total_interventions = df_detailed_results.groupby('defender_nfl_id').size().reset_index(name='total_interventions')
        df_with_players = df_with_players.merge(
            total_interventions,
            left_on='nfl_id',
            right_on='defender_nfl_id',
            how='left'
        )
        df_with_players['total_interventions'] = df_with_players['total_interventions'].fillna(0).astype(int)
        
        # Drop duplicate defender_nfl_id column
        if 'defender_nfl_id' in df_with_players.columns:
            df_with_players = df_with_players.drop('defender_nfl_id', axis=1)
    
    # ===== Create Interactive Widgets =====
    
    # Position filter dropdown
    position_dropdown = widgets.Dropdown(
        options=[
            ('All Positions', 'all'),
            ('LB (Linebackers)', 'LB'),
            ('CB (Cornerbacks)', 'CB'),
            ('S (Safeties)', 'S'),
            ('Other Positions', 'Other')
        ],
        value='all',
        description='Position:',
        style={'description_width': 'initial'},
        layout=Layout(width='250px')
    )
    
    # Plays filter dropdown
    plays_options = [
        ('No Filter', 0),
        ('≥ 20% of Max Plays', 20),
        ('≥ 50% of Max Plays', 50),
        ('≥ 80% of Max Plays', 80)
    ]
    
    max_plays = df_with_players['play_count'].max() if len(df_with_players) > 0 else 1
    plays_dropdown = widgets.Dropdown(
        options=plays_options,
        value=50,
        description='Play Volume:',
        style={'description_width': 'initial'},
        layout=Layout(width='250px')
    )
    
    # Build sort options dynamically based on intervention types
    sort_options = [
        ('Avg Attention (High to Low)', 'avg_attention'),
        ('High Attention % (High to Low)', 'high_attention_pct'),
        ('Play Count (High to Low)', 'play_count'),
        ('Impact (High to Low)', 'impact_removal'),
    ]

    
    # Add player name sort at the end
    sort_options.append(('Player Name (A-Z)', 'display_name'))
    
    # Sort options dropdown
    sort_dropdown = widgets.Dropdown(
        options=sort_options,
        value='impact_removal',
        description='Sort by:',
        style={'description_width': 'initial'},
        layout=Layout(width='300px')
    )
    
    # Results count label
    results_label = widgets.Label(value="")
    
    
    # Output area for the table
    table_output = widgets.Output()
    
    # ===== CSS Styles =====
    css_styles = """
    <style>
        .attention-table-container {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;
            background: linear-gradient(135deg, #0f172a 0%, #1a1a2e 100%);
            padding: 20px;
            border-radius: 12px;
            color: #fff;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            margin-top: 20px;
        }
        
        .table-header {
            text-align: center;
            margin-bottom: 25px;
            padding: 20px;
            background: linear-gradient(135deg, rgba(59, 130, 246, 0.15) 0%, rgba(6, 182, 212, 0.15) 100%);
            border-radius: 10px;
            border: 1px solid rgba(59, 130, 246, 0.3);
        }
        
        .header-icon {
            font-size: 36px;
            margin-bottom: 10px;
        }
        
        .header-text h1 {
            margin: 0;
            font-size: 28px;
            font-weight: bold;
            color: #fff;
            line-height: 1.3;
            background: linear-gradient(135deg, #3b82f6 0%, #06b6d4 100%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
        }
        
        .header-text p {
            margin: 8px 0 0 0;
            color: #9ca3af;
            font-size: 13px;
            font-weight: 500;
        }
        
        table {
            width: 100%;
            border-collapse: collapse;
            background: rgba(30, 30, 50, 0.5);
            backdrop-filter: blur(10px);
            border: 1px solid #374151;
            border-radius: 8px;
            overflow: visible;
            margin-top: 20px;
        }
        
        thead {
            background: linear-gradient(90deg, rgba(55, 65, 81, 0.8) 0%, rgba(55, 65, 81, 0.4) 100%);
            border-bottom: 2px solid #4b5563;
        }
        
        th {
            padding: 12px 8px;
            text-align: center;
            font-size: 11px;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 0.5px;
            color: #9ca3af;
        }
        
        td {
            padding: 10px 8px;
            border-bottom: 1px solid #2d3748;
            vertical-align: middle;
            text-align: center;
        }
        
        tbody tr {
            transition: all 0.2s ease;
        }
        
        tbody tr:hover {
            background: rgba(59, 130, 246, 0.1);
        }
        
        tbody tr:last-child td {
            border-bottom: none;
        }
        
        .rank-badge {
            display: inline-flex;
            align-items: center;
            justify-content: center;
            width: 32px;
            height: 32px;
            border-radius: 6px;
            background: #374151;
            font-weight: bold;
            font-size: 14px;
            position: relative;
        }
        
        .rank-badge.gold {
            background: #fbbf24;
            color: #78350f;
        }
        
        .rank-badge.silver {
            background: #d1d5db;
            color: #374151;
        }
        
        .rank-badge.bronze {
            background: #d97706;
            color: #fff;
        }
        
        .medal {
            position: absolute;
            top: -6px;
            right: -6px;
            font-size: 14px;
        }
        
        .player-cell {
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 8px;
            text-align: center;
        }
        
        .player-headshot {
            width: 32px;
            height: 32px;
            border-radius: 50%;
            object-fit: cover;
            border: 2px solid;
            flex-shrink: 0;
            box-shadow: 0 2px 8px rgba(0,0,0,0.3);
            position: relative;
        }
        
        .team-dot {
            position: absolute;
            bottom: -2px;
            right: -2px;
            width: 10px;
            height: 10px;
            border-radius: 50%;
            border: 2px solid #1f2937;
        }
        
        .player-info h3 {
            margin: 0;
            font-size: 11px;
            font-weight: 600;
            color: #fff;
            white-space: nowrap;
        }
        
        .player-info p {
            margin: 2px 0 0 0;
            font-size: 9px;
            color: #9ca3af;
            white-space: nowrap;
        }
        
        .position-badge {
            display: inline-block;
            background: #374151;
            color: #e5e7eb;
            padding: 2px 4px;
            border-radius: 3px;
            font-size: 8px;
            font-weight: 600;
            margin-right: 3px;
        }
        
        .team-badge {
            display: inline-block;
            color: #fff;
            padding: 2px 4px;
            border-radius: 3px;
            font-size: 8px;
            font-weight: 600;
        }
        
        .metric-label {
            font-size: 8px;
            color: #9ca3af;
            text-transform: uppercase;
            letter-spacing: 0.2px;
            margin-bottom: 2px;
        }
        
        .metric-value {
            font-size: 12px;
            font-weight: 700;
            color: #fff;
        }
        
        .attention-bar {
            width: 100%;
            height: 4px;
            background: #374151;
            border-radius: 2px;
            margin-top: 4px;
            overflow: hidden;
        }
        
        .attention-fill {
            height: 100%;
            border-radius: 2px;
            transition: width 0.3s ease;
        }
        
        .intervention-header {
            cursor: help;
            position: relative;
        }
        
        .legend-container {
            margin-top: 20px;
            padding: 15px;
            background: rgba(30, 30, 50, 0.5);
            border: 1px solid #374151;
            border-radius: 8px;
        }
        
        .legend-title {
            font-size: 12px;
            font-weight: 600;
            color: #e5e7eb;
            text-transform: uppercase;
            letter-spacing: 0.5px;
            margin-bottom: 10px;
        }
        
        .legend-item {
            margin-bottom: 8px;
            font-size: 10px;
            color: #d1d5db;
            line-height: 1.4;
        }
        
        .legend-item-title {
            font-weight: 600;
            color: #e5e7eb;
        }
        
        .legend-item-desc {
            color: #9ca3af;
            margin-top: 2px;
        }
        
        .no-results {
            padding: 40px;
            text-align: center;
            color: #9ca3af;
            font-size: 14px;
        }
    </style>
    """
    
    # ===== Helper Functions =====
    
    def create_table_html(filtered_df, sort_label):
        """Create HTML table for the filtered dataframe with bars for all metrics"""
        if len(filtered_df) == 0:
            return f"""
            {css_styles}
            <div class="attention-table-container">
                <div class="no-results">
                    No players match your current filters. Try adjusting the filter settings.
                </div>
            </div>
            """

        html_content = f"""
        {css_styles}
        <div class="attention-table-container">
            <div class="table-header">
                <div class="header-icon">🏈</div>
                <div class="header-text">
                    <h1>Neural Network Attention & Impact Analysis:<br>Top Defenders in 2023</h1>
                    <p>{len(filtered_df)} players displayed | Sorted by {sort_label}</p>
                </div>
            </div>
            
            <table>
                <thead>
                    <tr>
                        <th style="width: 5%;">Rank</th>
                        <th style="width: 40%;">Player</th>
                        <th style="width: 13%;">High Attn %</th>
                        <th style="width: 13%;">Avg Attention</th>
                        <th style="width: 29%;">REMOVAL<br><span style="font-size: 9px; text-transform: none; color: #6b7280;">Impact</span></th>
                    </tr>
                </thead>
                <tbody>
        """

        for idx, row in filtered_df.iterrows():
            rank = int(row['rank'])
            
            # Medal
            medal_class = ''
            medal_emoji = ''
            if rank == 1: medal_class, medal_emoji = 'gold', '🥇'
            elif rank == 2: medal_class, medal_emoji = 'silver', '🥈'
            elif rank == 3: medal_class, medal_emoji = 'bronze', '🥉'

            # Player info
            player_name = str(row.get('display_name', 'Unknown')).replace('<','&lt;').replace('>','&gt;')
            position = str(row.get('position', 'N/A')).replace('<','&lt;').replace('>','&gt;')
            team = str(row.get('latest_team', 'N/A')).replace('<','&lt;').replace('>','&gt;')
            headshot = row.get('headshot', '')
            team_color = row.get('team_color', '#374151')
            team_color2 = row.get('team_color2', '#6b7280')

            # Metrics
            avg_attention = float(row.get('avg_attention', 0)) *100
            high_attention_pct = float(row.get('high_attention_pct', 0))
            avg_impact = float(row.get('impact_removal', 0)) *100
            play_count = int(row.get('play_count', 0))

            # Bar widths
            high_attention_width = min(high_attention_pct, 100)
            avg_attention_width = min(avg_attention * 100, 100)
            max_impact = df_detailed_results['impact_score'].max() / 0.07
            impact_width = min((avg_impact / max_impact) * 100, 100) if max_impact > 0 else 0


            # Headshot HTML
            if pd.notna(headshot) and headshot and isinstance(headshot, str):
                headshot_html = f'<img src="{headshot}" alt="{player_name}" class="player-headshot" style="border-color: {team_color};" onerror="this.style.display=\'none\';">'
            else:
                initials = ''.join([w[0].upper() for w in player_name.split() if w])[:2] or '??'
                headshot_html = f'<div class="player-headshot" style="border-color: {team_color}; background: linear-gradient(135deg, {team_color}, {team_color2}); display: flex; align-items: center; justify-content: center; font-weight: bold; font-size: 12px; color: #fff;">{initials}</div>'

            html_content += f"""
                <tr>
                    <td>
                        <div class="rank-badge {medal_class}">
                            {rank}{f'<span class="medal">{medal_emoji}</span>' if medal_emoji else ''}
                        </div>
                    </td>
                    <td>
                        <div class="player-cell">
                            <div style="position: relative;">
                                {headshot_html}
                                <div class="team-dot" style="background-color: {team_color2};"></div>
                            </div>
                            <div class="player-info">
                                <h3>{player_name}</h3>
                                <p>
                                    <span class="position-badge">{position}</span>
                                    <span class="team-badge" style="background-color: {team_color};">{team}</span>
                                    <span style="color: #6b7280; font-size: 8px;">{play_count} plays</span>
                                </p>
                            </div>
                        </div>
                    </td>

                    <!-- High Attention % with bar -->
                    <td>
                        <div class="metric-label">High Attention</div>
                        <div class="metric-value">{high_attention_pct:.1f}%</div>
                        <div class="attention-bar">
                            <div class="attention-fill" style="width: {high_attention_width:.1f}%; background: linear-gradient(90deg, {team_color}, {team_color2});"></div>
                        </div>
                    </td>

                    <!-- Avg Attention with bar -->
                    <td>
                        <div class="metric-label">Avg Attention</div>
                        <div class="metric-value">{avg_attention:.1f}%</div>
                        <div class="attention-bar">
                            <div class="attention-fill" style="width: {avg_attention_width:.1f}%; background: linear-gradient(90deg, {team_color}, {team_color2});"></div>
                        </div>
                    </td>

                    <!-- Impact Removal with bar -->
                    <td>
                        <div class="metric-label">Impact</div>
                        <div class="metric-value">{avg_impact:.1f}%</div>
                        <div class="attention-bar">
                            <div class="attention-fill" style="width: {impact_width:.1f}%; background: linear-gradient(90deg, {team_color}, {team_color2});"></div>
                        </div>
                    </td>
                </tr>
            """

        html_content += """
                </tbody>
            </table>
        </div>
        """
        return html_content

    
    
    def update_table(*args):
        """Update the table based on current filter values"""
        with table_output:
            clear_output(wait=True)
            
            # Apply filters
            filtered_df = df_with_players.copy()
            
            # Position filter
            if position_dropdown.value != 'all':
                filtered_df = filtered_df[filtered_df['position_group'] == position_dropdown.value]
            
            # Plays filter
            if plays_dropdown.value > 0:
                current_max_plays = filtered_df['play_count'].max() if len(filtered_df) > 0 else 1
                threshold = (plays_dropdown.value / 100) * current_max_plays
                filtered_df = filtered_df[filtered_df['play_count'] >= threshold]
            
            # Apply sorting
            sort_column = sort_dropdown.value
            ascending = sort_column == 'display_name'  # Sort A-Z for names, else descending
            
            if sort_column in filtered_df.columns:
                filtered_df = filtered_df.sort_values(sort_column, ascending=ascending)
            
            # Add rank based on sort
            filtered_df = filtered_df.copy()
            filtered_df['rank'] = range(1, len(filtered_df) + 1)
            
            # Update results label
            results_label.value = f"Showing {len(filtered_df)} of {len(df_with_players)} players"
            
            # Get the current sort label for display
            sort_label = dict(sort_dropdown.options).get(sort_dropdown.value, sort_dropdown.value)
            
            # Display table
            display(HTML(create_table_html(filtered_df, sort_label)))
    

    
    # ===== Setup Event Handlers =====
    position_dropdown.observe(update_table, names='value')
    plays_dropdown.observe(update_table, names='value')
    sort_dropdown.observe(update_table, names='value')
    
    # ===== Create Layout =====
    filters_row = HBox([
        position_dropdown,
        plays_dropdown,
        sort_dropdown
    ], layout=Layout(
        display='flex',
        justify_content='space-between',
        margin='10px 0',
        background = 'linear-gradient(135deg, #0f172a 0%, #1a1a2e 100%)'

    ))
    
    controls_row = HBox([
        results_label
    ], layout=Layout(
        display='flex',
        justify_content='space-between',
        align_items='center',
        margin='10px 0 20px 0'
    ))
    
    # Initial results label
    results_label.value = f"Total players: {len(df_with_players)}"
    
    # ===== Initial Display =====
    # Create main container
    main_container = VBox([
        filters_row,
        controls_row,
        table_output
    ], layout=Layout(
        width='100%',
        padding='20px',
        background='linear-gradient(135deg, #0f172a 0%, #1a1a2e 100%)',
        border_radius='12px',
        box_shadow='0 20px 60px rgba(0,0,0,0.3)'
    ))
    
    # Update table with initial data
    update_table()
    
    return main_container


# ===== USAGE EXAMPLE =====
# In a Jupyter notebook cell:
#
# from player_attention_table_improved import create_player_attention_table
# from IPython.display import display
#
# # Assuming you have df, nfl_players, nfl_teams, and df_detailed_results loaded
# interactive_table = create_player_attention_table(df, nfl_players, nfl_teams, df_detailed_results)
# display(interactive_table)

In [106]:
interactive_table = create_player_attention_table(df, nfl_players, nfl_teams, df_detailed_results)
display(interactive_table)